# Classificacao da Qualidade de Vinhos com Machine Learning

## Tech Challenge - Fase 2 | POSTECH FIAP

**Objetivo:** Desenvolver um modelo de classificacao binaria capaz de prever a qualidade de um vinho (Alta vs Baixa/Media) com base em suas caracteristicas fisico-quimicas.

**Dataset:** Wine Quality Dataset (Kaggle) - 1.143 amostras de vinhos tintos com 11 variaveis fisico-quimicas e 1 variavel alvo (quality).

---

## Sumario

1. [Compreensao do Problema](#1-compreensao-do-problema)
2. [Analise Exploratoria de Dados (EDA)](#2-analise-exploratoria-de-dados-eda)
3. [Pre-processamento de Dados](#3-pre-processamento-de-dados)
4. [Desenvolvimento de Modelos](#4-desenvolvimento-de-modelos)
5. [Avaliacao dos Modelos](#5-avaliacao-dos-modelos)
6. [Interpretacao dos Resultados](#6-interpretacao-dos-resultados)


---

# 1. Compreensao do Problema

A industria vitivinicola depende tradicionalmente de avaliacao sensorial por especialistas para determinar a qualidade dos vinhos. Este processo e subjetivo, demorado e caro.

**Proposta:** Utilizar tecnicas de Machine Learning para classificar automaticamente a qualidade dos vinhos a partir de analises fisico-quimicas, permitindo:
- Padronizacao da avaliacao de qualidade
- Reducao de custos com paineis de degustacao
- Auxilio ao enologo na tomada de decisao durante a producao

**Variavel Alvo (Classificacao Binaria):**
- **Alta Qualidade**: nota >= 7
- **Baixa/Media Qualidade**: nota < 7


In [ ]:
import os
os.chdir('/home/ubuntu/wine-quality-classification')

# Importacoes
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Configuracoes de visualizacao
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

import sys
sys.path.insert(0, '.')
from src.preprocessing import (
    load_data, create_binary_target, remove_id_column,
    get_features_and_target, scale_features, apply_smote, split_data
)
from src.modeling import (
    get_models, train_model, evaluate_model,
    cross_validate_model, get_feature_importance
)

print('Bibliotecas carregadas com sucesso!')

Bibliotecas carregadas com sucesso!


In [ ]:
# 1.1 Carregamento dos Dados
df_raw = load_data('data/WineQT.csv')
print(f'Dataset carregado: {df_raw.shape[0]} amostras, {df_raw.shape[1]} colunas')
print(f'\nColunas: {list(df_raw.columns)}')
df_raw.head(10)

Dataset carregado: 1143 amostras, 13 colunas

Colunas: ['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar', 'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density', 'pH', 'sulphates', 'alcohol', 'quality', 'Id']


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,Id
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,0
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,1
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,2
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6,3
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,4
5,7.4,0.66,0.00,1.8,0.075,13.0,40.0,0.9978,3.51,0.56,9.4,5,5
6,7.9,0.60,0.06,1.6,0.069,15.0,59.0,0.9964,3.30,0.46,9.4,5,6
7,7.3,0.65,0.00,1.2,0.065,15.0,21.0,0.9946,3.39,0.47,10.0,7,7
8,7.8,0.58,0.02,2.0,0.073,9.0,18.0,0.9968,3.36,0.57,9.5,7,8
9,6.7,0.58,0.08,1.8,0.097,15.0,65.0,0.9959,3.28,0.54,9.2,5,10


In [ ]:
# 1.2 Informacoes gerais do dataset
print('=' * 60)
print('INFORMACOES DO DATASET')
print('=' * 60)
df_raw.info()
print('\n')
print('Estatisticas descritivas:')
df_raw.describe().round(3)

INFORMACOES DO DATASET
<class 'pandas.DataFrame'>
RangeIndex: 1143 entries, 0 to 1142
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1143 non-null   float64
 1   volatile acidity      1143 non-null   float64
 2   citric acid           1143 non-null   float64
 3   residual sugar        1143 non-null   float64
 4   chlorides             1143 non-null   float64
 5   free sulfur dioxide   1143 non-null   float64
 6   total sulfur dioxide  1143 non-null   float64
 7   density               1143 non-null   float64
 8   pH                    1143 non-null   float64
 9   sulphates             1143 non-null   float64
 10  alcohol               1143 non-null   float64
 11  quality               1143 non-null   int64  
 12  Id                    1143 non-null   int64  
dtypes: float64(11), int64(2)
memory usage: 116.2 KB


Estatisticas descritivas:


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,Id
count,1143.000,1143.000,1143.000,1143.000,1143.000,1143.000,1143.000,1143.000,1143.000,1143.000,1143.000,1143.000,1143.000
mean,8.311,0.531,0.268,2.532,0.087,15.615,45.915,0.997,3.311,0.658,10.442,5.657,804.969
std,1.748,0.180,0.197,1.356,0.047,10.250,32.782,0.002,0.157,0.170,1.082,0.806,463.997
min,4.600,0.120,0.000,0.900,0.012,1.000,6.000,0.990,2.740,0.330,8.400,3.000,0.000
25%,7.100,0.392,0.090,1.900,0.070,7.000,21.000,0.996,3.205,0.550,9.500,5.000,411.000
50%,7.900,0.520,0.250,2.200,0.079,13.000,37.000,0.997,3.310,0.620,10.200,6.000,794.000
75%,9.100,0.640,0.420,2.600,0.090,21.000,61.000,0.998,3.400,0.730,11.100,6.000,1209.500
max,15.900,1.580,1.000,15.500,0.611,68.000,289.000,1.004,4.010,2.000,14.900,8.000,1597.000


In [ ]:
# 1.3 Remover coluna Id (apenas identificador, nao tem valor preditivo)
df = remove_id_column(df_raw)
print(f'Colunas apos remocao do Id: {list(df.columns)}')
print(f'Shape: {df.shape}')

Colunas apos remocao do Id: ['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar', 'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density', 'pH', 'sulphates', 'alcohol', 'quality']
Shape: (1143, 12)


In [ ]:
# 1.4 Distribuicao original da variavel quality
print('Distribuicao da variavel quality (original):')
print(df['quality'].value_counts().sort_index())
print(f'\nMedia: {df["quality"].mean():.2f}')
print(f'Mediana: {df["quality"].median():.1f}')
print(f'Desvio padrao: {df["quality"].std():.2f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma
ax1 = axes[0]
quality_counts = df['quality'].value_counts().sort_index()
colors = ['#e74c3c' if q < 7 else '#27ae60' for q in quality_counts.index]
ax1.bar(quality_counts.index, quality_counts.values, color=colors, edgecolor='black', alpha=0.8)
ax1.set_xlabel('Nota de Qualidade', fontsize=13)
ax1.set_ylabel('Quantidade de Amostras', fontsize=13)
ax1.set_title('Distribuicao das Notas de Qualidade', fontsize=14, fontweight='bold')
for i, (q, c) in enumerate(zip(quality_counts.index, quality_counts.values)):
    ax1.text(q, c + 5, str(c), ha='center', fontweight='bold', fontsize=11)

# Pie chart binario
ax2 = axes[1]
df_temp = create_binary_target(df)
binary_counts = df_temp['quality_label'].value_counts()
labels = ['Baixa/Media\nQualidade (< 7)', 'Alta Qualidade\n(>= 7)']
colors_pie = ['#e74c3c', '#27ae60']
wedges, texts, autotexts = ax2.pie(
    binary_counts.values, labels=labels, colors=colors_pie,
    autopct='%1.1f%%', startangle=90, textprops={'fontsize': 12},
    explode=(0, 0.05)
)
for t in autotexts:
    t.set_fontweight('bold')
    t.set_fontsize(13)
ax2.set_title('Classificacao Binaria', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('results/01_distribuicao_quality.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nGrafico salvo em results/01_distribuicao_quality.png')

Distribuicao da variavel quality (original):
quality
3      6
4     33
5    483
6    462
7    143
8     16
Name: count, dtype: int64

Media: 5.66
Mediana: 6.0
Desvio padrao: 0.81



Grafico salvo em results/01_distribuicao_quality.png


In [ ]:
# 1.5 Transformacao em classificacao binaria
df = create_binary_target(df)
print('Variavel quality_label criada:')
print(f'  0 (Baixa/Media Qualidade - nota < 7): {(df["quality_label"] == 0).sum()} amostras ({(df["quality_label"] == 0).mean()*100:.1f}%)')
print(f'  1 (Alta Qualidade - nota >= 7):        {(df["quality_label"] == 1).sum()} amostras ({(df["quality_label"] == 1).mean()*100:.1f}%)')
print(f'\n>>> ALERTA: Classes fortemente desbalanceadas! Proporcao ~6:1')
print('>>> Sera necessario tratar o desbalanceamento para evitar modelos enviesados.')

Variavel quality_label criada:
  0 (Baixa/Media Qualidade - nota < 7): 984 amostras (86.1%)
  1 (Alta Qualidade - nota >= 7):        159 amostras (13.9%)

>>> ALERTA: Classes fortemente desbalanceadas! Proporcao ~6:1
>>> Sera necessario tratar o desbalanceamento para evitar modelos enviesados.


---

# 2. Analise Exploratoria de Dados (EDA)

Nesta etapa, investigaremos as distribuicoes das variaveis, correlacoes, outliers e o balanceamento das classes.


In [ ]:
# 2.1 Verificacao de dados faltantes
print('=' * 60)
print('VERIFICACAO DE DADOS FALTANTES')
print('=' * 60)
missing = df.isnull().sum()
print(missing)
print(f'\nTotal de valores faltantes: {missing.sum()}')
print('>>> Nenhum valor faltante encontrado. O dataset esta completo.')

VERIFICACAO DE DADOS FALTANTES
fixed acidity           0
volatile acidity        0
citric acid             0
residual sugar          0
chlorides               0
free sulfur dioxide     0
total sulfur dioxide    0
density                 0
pH                      0
sulphates               0
alcohol                 0
quality                 0
quality_label           0
dtype: int64

Total de valores faltantes: 0
>>> Nenhum valor faltante encontrado. O dataset esta completo.


In [ ]:
# 2.2 Distribuicao de todas as variaveis (histogramas com KDE)
feature_cols = [c for c in df.columns if c not in ['quality', 'quality_label']]

fig, axes = plt.subplots(4, 3, figsize=(18, 20))
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    ax = axes[i]
    ax.hist(df[col], bins=30, color='#3498db', alpha=0.7, edgecolor='black', density=True)
    df[col].plot.kde(ax=ax, color='#e74c3c', linewidth=2)
    ax.set_title(col, fontsize=13, fontweight='bold')
    ax.set_xlabel('')
    mean_val = df[col].mean()
    median_val = df[col].median()
    ax.axvline(mean_val, color='#e74c3c', linestyle='--', linewidth=1.5, label=f'Media: {mean_val:.2f}')
    ax.axvline(median_val, color='#2ecc71', linestyle='-.', linewidth=1.5, label=f'Mediana: {median_val:.2f}')
    ax.legend(fontsize=9)

if len(feature_cols) < len(axes):
    for j in range(len(feature_cols), len(axes)):
        fig.delaxes(axes[j])

plt.suptitle('Distribuicao das Variaveis Fisico-Quimicas', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('results/02_distribuicao_variaveis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafico salvo em results/02_distribuicao_variaveis.png')

Grafico salvo em results/02_distribuicao_variaveis.png


In [ ]:
# 2.3 Boxplots para deteccao de outliers
fig, axes = plt.subplots(4, 3, figsize=(18, 20))
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    ax = axes[i]
    bp = ax.boxplot(df[col], patch_artist=True, vert=True,
                     boxprops=dict(facecolor='#3498db', alpha=0.7),
                     medianprops=dict(color='#e74c3c', linewidth=2),
                     whiskerprops=dict(linewidth=1.5),
                     flierprops=dict(marker='o', markerfacecolor='#e74c3c', markersize=5))
    ax.set_title(col, fontsize=13, fontweight='bold')
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    ax.set_xlabel(f'Outliers: {outliers} ({outliers/len(df)*100:.1f}%)', fontsize=10, color='red')

if len(feature_cols) < len(axes):
    for j in range(len(feature_cols), len(axes)):
        fig.delaxes(axes[j])

plt.suptitle('Boxplots - Deteccao de Outliers (Metodo IQR)', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('results/03_boxplots_outliers.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafico salvo em results/03_boxplots_outliers.png')

Grafico salvo em results/03_boxplots_outliers.png


In [ ]:
# 2.4 Resumo quantitativo de outliers
print('=' * 70)
print('DETECCAO DE OUTLIERS (Metodo IQR - Interquartile Range)')
print('=' * 70)
outlier_summary = []
for col in feature_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    outlier_summary.append({
        'Variavel': col,
        'Q1': round(Q1, 3),
        'Q3': round(Q3, 3),
        'IQR': round(IQR, 3),
        'Limite Inferior': round(lower, 3),
        'Limite Superior': round(upper, 3),
        'N Outliers': n_outliers,
        '% Outliers': round(n_outliers / len(df) * 100, 1)
    })

outlier_df = pd.DataFrame(outlier_summary)
print(outlier_df.to_string(index=False))
print(f'\n>>> As variaveis com mais outliers sao indicativas de vinhos com composicoes extremas.')
print('>>> Optamos por NAO remover outliers, pois podem representar vinhos de alta qualidade.')

DETECCAO DE OUTLIERS (Metodo IQR - Interquartile Range)
            Variavel     Q1     Q3    IQR  Limite Inferior  Limite Superior  N Outliers  % Outliers
       fixed acidity  7.100  9.100  2.000            4.100           12.100          44         3.8
    volatile acidity  0.392  0.640  0.248            0.021            1.011          14         1.2
         citric acid  0.090  0.420  0.330           -0.405            0.915           1         0.1
      residual sugar  1.900  2.600  0.700            0.850            3.650         110         9.6
           chlorides  0.070  0.090  0.020            0.040            0.120          77         6.7
 free sulfur dioxide  7.000 21.000 14.000          -14.000           42.000          18         1.6
total sulfur dioxide 21.000 61.000 40.000          -39.000          121.000          40         3.5
             density  0.996  0.998  0.002            0.992            1.001          36         3.1
                  pH  3.205  3.400  0.195   

In [ ]:
# 2.5 Matriz de correlacao
fig, ax = plt.subplots(figsize=(14, 10))
corr_matrix = df[feature_cols].corr()

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
    center=0, vmin=-1, vmax=1, square=True, linewidths=0.5,
    cbar_kws={'shrink': 0.8, 'label': 'Correlacao de Pearson'},
    ax=ax
)
ax.set_title('Matriz de Correlacao entre Variaveis Fisico-Quimicas', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('results/04_matriz_correlacao.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafico salvo em results/04_matriz_correlacao.png')

Grafico salvo em results/04_matriz_correlacao.png


In [ ]:
# 2.6 Analise detalhada das correlacoes mais relevantes
print('=' * 70)
print('ANALISE DAS CORRELACOES MAIS SIGNIFICATIVAS')
print('=' * 70)

corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        r = corr_matrix.iloc[i, j]
        if abs(r) > 0.3:
            corr_pairs.append((corr_matrix.columns[i], corr_matrix.columns[j], r))

corr_pairs.sort(key=lambda x: abs(x[2]), reverse=True)

for var1, var2, r in corr_pairs:
    strength = 'FORTE' if abs(r) > 0.6 else 'MODERADA'
    direction = 'POSITIVA' if r > 0 else 'NEGATIVA'
    print(f'  {var1} vs {var2}: r = {r:.3f} ({strength} {direction})')

print('\n--- Justificativas ---')
justificativas = [
    '1. fixed acidity vs citric acid (r positivo): Ambos sao acidos presentes no vinho;',
    '   maior acidez fixa tende a acompanhar maior presenca de acido citrico.',
    '',
    '2. fixed acidity vs density (r positivo): Acidos dissolvidos aumentam a densidade',
    '   do vinho, relacao fisico-quimica esperada.',
    '',
    '3. fixed acidity vs pH (r negativo): Relacao quimica direta - maior concentracao',
    '   de acido (acidez fixa) resulta em menor pH (mais acido).',
    '',
    '4. free sulfur dioxide vs total sulfur dioxide (r positivo): O SO2 livre e um',
    '   subconjunto do SO2 total, logo a correlacao e esperada.',
    '',
    '5. density vs alcohol (r negativo): O alcool tem densidade menor que a agua,',
    '   portanto maior teor alcoolico reduz a densidade do vinho.',
    '',
    '6. citric acid vs volatile acidity (r negativo): Durante a fermentacao,',
    '   a volatilidade da acidez tende a reduzir com maior presenca de acido citrico.',
]
for line in justificativas:
    print(line)

ANALISE DAS CORRELACOES MAIS SIGNIFICATIVAS
  fixed acidity vs pH: r = -0.685 (FORTE NEGATIVA)
  fixed acidity vs density: r = 0.682 (FORTE POSITIVA)
  fixed acidity vs citric acid: r = 0.673 (FORTE POSITIVA)
  free sulfur dioxide vs total sulfur dioxide: r = 0.661 (FORTE POSITIVA)
  citric acid vs pH: r = -0.546 (MODERADA NEGATIVA)
  volatile acidity vs citric acid: r = -0.544 (MODERADA NEGATIVA)
  density vs alcohol: r = -0.495 (MODERADA NEGATIVA)
  residual sugar vs density: r = 0.380 (MODERADA POSITIVA)
  citric acid vs density: r = 0.375 (MODERADA POSITIVA)
  chlorides vs sulphates: r = 0.375 (MODERADA POSITIVA)
  density vs pH: r = -0.353 (MODERADA NEGATIVA)
  citric acid vs sulphates: r = 0.331 (MODERADA POSITIVA)

--- Justificativas ---
1. fixed acidity vs citric acid (r positivo): Ambos sao acidos presentes no vinho;
   maior acidez fixa tende a acompanhar maior presenca de acido citrico.

2. fixed acidity vs density (r positivo): Acidos dissolvidos aumentam a densidade
   do 

In [ ]:
# 2.7 Correlacao das variaveis com a qualidade
fig, ax = plt.subplots(figsize=(12, 6))

corr_with_quality = df[feature_cols + ['quality_label']].corr()['quality_label'].drop('quality_label')
corr_sorted = corr_with_quality.sort_values(ascending=True)

colors = ['#27ae60' if v > 0 else '#e74c3c' for v in corr_sorted.values]
bars = ax.barh(corr_sorted.index, corr_sorted.values, color=colors, edgecolor='black', alpha=0.8)
ax.set_xlabel('Correlacao com Qualidade (Binaria)', fontsize=13)
ax.set_title('Correlacao de Cada Variavel com a Qualidade do Vinho', fontsize=14, fontweight='bold')
ax.axvline(x=0, color='black', linewidth=0.8)

for bar, val in zip(bars, corr_sorted.values):
    ax.text(val + 0.01 if val > 0 else val - 0.01, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', ha='left' if val > 0 else 'right', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('results/05_correlacao_com_qualidade.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafico salvo em results/05_correlacao_com_qualidade.png')

print('\n--- Principais influenciadores da qualidade ---')
print(f'  Maior correlacao POSITIVA: {corr_sorted.index[-1]} ({corr_sorted.values[-1]:.3f})')
print(f'  Maior correlacao NEGATIVA: {corr_sorted.index[0]} ({corr_sorted.values[0]:.3f})')

Grafico salvo em results/05_correlacao_com_qualidade.png

--- Principais influenciadores da qualidade ---
  Maior correlacao POSITIVA: alcohol (0.404)
  Maior correlacao NEGATIVA: volatile acidity (-0.305)


In [ ]:
# 2.8 Distribuicao das variaveis por classe de qualidade
fig, axes = plt.subplots(4, 3, figsize=(18, 20))
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    ax = axes[i]
    df[df['quality_label'] == 0][col].hist(
        ax=ax, bins=25, alpha=0.6, color='#e74c3c', label='Baixa/Media', density=True, edgecolor='black'
    )
    df[df['quality_label'] == 1][col].hist(
        ax=ax, bins=25, alpha=0.6, color='#27ae60', label='Alta', density=True, edgecolor='black'
    )
    ax.set_title(col, fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)

if len(feature_cols) < len(axes):
    for j in range(len(feature_cols), len(axes)):
        fig.delaxes(axes[j])

plt.suptitle('Distribuicao das Variaveis por Classe de Qualidade', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('results/06_distribuicao_por_classe.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafico salvo em results/06_distribuicao_por_classe.png')

Grafico salvo em results/06_distribuicao_por_classe.png


In [ ]:
# 2.9 Teste estatistico - Diferenca entre as classes
print('=' * 70)
print('TESTE DE MANN-WHITNEY U - Diferenca significativa entre classes')
print('=' * 70)

for col in feature_cols:
    group_low = df[df['quality_label'] == 0][col]
    group_high = df[df['quality_label'] == 1][col]
    stat, p_value = stats.mannwhitneyu(group_low, group_high, alternative='two-sided')
    sig = '***' if p_value < 0.001 else ('**' if p_value < 0.01 else ('*' if p_value < 0.05 else 'ns'))
    print(f'  {col:25s} | p-value: {p_value:.2e} | {sig}')

print('\n  *** p < 0.001  |  ** p < 0.01  |  * p < 0.05  |  ns = nao significativo')
print('\n>>> Variaveis com diferenca significativa entre classes sao potenciais bons preditores.')

TESTE DE MANN-WHITNEY U - Diferenca significativa entre classes
  fixed acidity             | p-value: 3.43e-05 | ***
  volatile acidity          | p-value: 3.19e-28 | ***
  citric acid               | p-value: 1.02e-16 | ***
  residual sugar            | p-value: 4.14e-02 | *
  chlorides                 | p-value: 2.60e-09 | ***
  free sulfur dioxide       | p-value: 2.94e-02 | *
  total sulfur dioxide      | p-value: 8.74e-07 | ***
  density                   | p-value: 2.24e-07 | ***
  pH                        | p-value: 6.47e-03 | **
  sulphates                 | p-value: 8.00e-23 | ***
  alcohol                   | p-value: 2.78e-38 | ***

  *** p < 0.001  |  ** p < 0.01  |  * p < 0.05  |  ns = nao significativo

>>> Variaveis com diferenca significativa entre classes sao potenciais bons preditores.


---

# 3. Pre-processamento de Dados

Etapas de preparacao dos dados para modelagem:
1. Separacao de features e variavel alvo
2. Divisao treino/teste (80/20) com estratificacao
3. Normalizacao com StandardScaler
4. Balanceamento de classes com SMOTE


In [ ]:
# 3.1 Separacao de features e target
X, y = get_features_and_target(df)
print(f'Features (X): {X.shape}')
print(f'Target (y): {y.shape}')
print(f'\nFeatures utilizadas: {list(X.columns)}')

Features (X): (1143, 11)
Target (y): (1143,)

Features utilizadas: ['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar', 'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density', 'pH', 'sulphates', 'alcohol']


In [ ]:
# 3.2 Divisao treino/teste (80/20) com estratificacao
X_train, X_test, y_train, y_test = split_data(X, y, test_size=0.2, random_state=42)

print(f'Conjunto de TREINO: {X_train.shape[0]} amostras')
print(f'  - Baixa/Media qualidade: {(y_train == 0).sum()} ({(y_train == 0).mean()*100:.1f}%)')
print(f'  - Alta qualidade:        {(y_train == 1).sum()} ({(y_train == 1).mean()*100:.1f}%)')
print(f'\nConjunto de TESTE: {X_test.shape[0]} amostras')
print(f'  - Baixa/Media qualidade: {(y_test == 0).sum()} ({(y_test == 0).mean()*100:.1f}%)')
print(f'  - Alta qualidade:        {(y_test == 1).sum()} ({(y_test == 1).mean()*100:.1f}%)')
print('\n>>> Estratificacao garante mesma proporcao de classes em treino e teste.')

Conjunto de TREINO: 914 amostras
  - Baixa/Media qualidade: 787 (86.1%)
  - Alta qualidade:        127 (13.9%)

Conjunto de TESTE: 229 amostras
  - Baixa/Media qualidade: 197 (86.0%)
  - Alta qualidade:        32 (14.0%)

>>> Estratificacao garante mesma proporcao de classes em treino e teste.


In [ ]:
# 3.3 Normalizacao das features com StandardScaler
X_train_scaled, X_test_scaled, scaler = scale_features(X_train, X_test)

print('StandardScaler aplicado.')
print('\nMedias apos normalizacao (treino - devem ser ~0):')
print(X_train_scaled.mean().round(6))
print('\nDesvios padrao apos normalizacao (treino - devem ser ~1):')
print(X_train_scaled.std().round(6))

StandardScaler aplicado.

Medias apos normalizacao (treino - devem ser ~0):
fixed acidity          -0.0
volatile acidity       -0.0
citric acid            -0.0
residual sugar          0.0
chlorides              -0.0
free sulfur dioxide    -0.0
total sulfur dioxide   -0.0
density                 0.0
pH                     -0.0
sulphates               0.0
alcohol                -0.0
dtype: float64

Desvios padrao apos normalizacao (treino - devem ser ~1):
fixed acidity           1.000547
volatile acidity        1.000547
citric acid             1.000547
residual sugar          1.000547
chlorides               1.000547
free sulfur dioxide     1.000547
total sulfur dioxide    1.000547
density                 1.000547
pH                      1.000547
sulphates               1.000547
alcohol                 1.000547
dtype: float64


In [ ]:
# 3.4 Balanceamento de classes com SMOTE
print('ANTES do SMOTE:')
print(f'  Classe 0 (Baixa/Media): {(y_train == 0).sum()}')
print(f'  Classe 1 (Alta):        {(y_train == 1).sum()}')

X_train_balanced, y_train_balanced = apply_smote(X_train_scaled, y_train, random_state=42)

print(f'\nDEPOIS do SMOTE:')
print(f'  Classe 0 (Baixa/Media): {(y_train_balanced == 0).sum()}')
print(f'  Classe 1 (Alta):        {(y_train_balanced == 1).sum()}')
print('\n>>> SMOTE gera amostras sinteticas da classe minoritaria para balancear o treinamento.')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(['Baixa/Media', 'Alta'], [int((y_train == 0).sum()), int((y_train == 1).sum())],
            color=['#e74c3c', '#27ae60'], edgecolor='black')
axes[0].set_title('Antes do SMOTE', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Quantidade')

axes[1].bar(['Baixa/Media', 'Alta'], [int((y_train_balanced == 0).sum()), int((y_train_balanced == 1).sum())],
            color=['#e74c3c', '#27ae60'], edgecolor='black')
axes[1].set_title('Depois do SMOTE', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Quantidade')

plt.suptitle('Balanceamento de Classes com SMOTE', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('results/07_smote_balanceamento.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafico salvo em results/07_smote_balanceamento.png')

ANTES do SMOTE:
  Classe 0 (Baixa/Media): 787
  Classe 1 (Alta):        127

DEPOIS do SMOTE:
  Classe 0 (Baixa/Media): 787
  Classe 1 (Alta):        787

>>> SMOTE gera amostras sinteticas da classe minoritaria para balancear o treinamento.


Grafico salvo em results/07_smote_balanceamento.png


---

# 4. Desenvolvimento de Modelos

Serao treinados e comparados 4 modelos de classificacao:
1. **Logistic Regression** - Modelo linear, interpretavel
2. **Random Forest** - Ensemble de arvores de decisao
3. **XGBoost** - Gradient boosting otimizado
4. **Gradient Boosting** - Ensemble sequencial de arvores


In [ ]:
# 4.1 Treinamento dos modelos
models = get_models()
trained_models = {}
results = {}

print('=' * 70)
print('TREINAMENTO DOS MODELOS')
print('=' * 70)

for name, model in models.items():
    print(f'\nTreinando {name}...')
    trained = train_model(model, X_train_balanced, y_train_balanced)
    trained_models[name] = trained
    metrics, y_pred, y_proba = evaluate_model(trained, X_test_scaled, y_test)
    results[name] = {'metrics': metrics, 'y_pred': y_pred, 'y_proba': y_proba}
    print(f'  Accuracy:  {metrics["Accuracy"]:.4f}')
    print(f'  Precision: {metrics["Precision"]:.4f}')
    print(f'  Recall:    {metrics["Recall"]:.4f}')
    print(f'  F1-Score:  {metrics["F1-Score"]:.4f}')
    print(f'  AUC-ROC:   {metrics["AUC-ROC"]:.4f}')

print('\n>>> Todos os modelos treinados com sucesso!')

TREINAMENTO DOS MODELOS

Treinando Logistic Regression...
  Accuracy:  0.7904
  Precision: 0.3621
  Recall:    0.6562
  F1-Score:  0.4667
  AUC-ROC:   0.8484

Treinando Random Forest...


  Accuracy:  0.8777
  Precision: 0.5556
  Recall:    0.6250
  F1-Score:  0.5882
  AUC-ROC:   0.9067

Treinando XGBoost...
  Accuracy:  0.8865
  Precision: 0.5789
  Recall:    0.6875
  F1-Score:  0.6286
  AUC-ROC:   0.8907

Treinando Gradient Boosting...


  Accuracy:  0.8908
  Precision: 0.5946
  Recall:    0.6875
  F1-Score:  0.6377
  AUC-ROC:   0.8614

>>> Todos os modelos treinados com sucesso!


In [ ]:
# 4.2 Validacao Cruzada (5-Fold) para robustez
print('=' * 70)
print('VALIDACAO CRUZADA (5-Fold) - F1-Score')
print('=' * 70)

cv_results = {}
for name, model in models.items():
    scores = cross_validate_model(model, X_train_balanced, y_train_balanced, cv=5)
    cv_results[name] = scores
    print(f'\n{name}:')
    print(f'  F1 por fold: {[f"{s:.4f}" for s in scores]}')
    print(f'  Media:       {scores.mean():.4f} (+/- {scores.std():.4f})')

VALIDACAO CRUZADA (5-Fold) - F1-Score

Logistic Regression:
  F1 por fold: ['0.8052', '0.8148', '0.8288', '0.8457', '0.8536']
  Media:       0.8296 (+/- 0.0182)



Random Forest:
  F1 por fold: ['0.9304', '0.9422', '0.9366', '0.9541', '0.9627']
  Media:       0.9452 (+/- 0.0117)



XGBoost:
  F1 por fold: ['0.9264', '0.9401', '0.9341', '0.9483', '0.9536']
  Media:       0.9405 (+/- 0.0097)



Gradient Boosting:
  F1 por fold: ['0.8918', '0.9281', '0.9366', '0.9383', '0.9530']
  Media:       0.9296 (+/- 0.0205)


---

# 5. Avaliacao dos Modelos

Comparacao detalhada dos modelos utilizando multiplas metricas, matrizes de confusao e curvas ROC.


In [ ]:
# 5.1 Tabela comparativa de metricas
metrics_df = pd.DataFrame({name: res['metrics'] for name, res in results.items()}).T
metrics_df = metrics_df.round(4)
metrics_df = metrics_df.sort_values('F1-Score', ascending=False)

print('=' * 70)
print('COMPARACAO DE METRICAS DOS MODELOS')
print('=' * 70)
print(metrics_df.to_string())
print(f'\n>>> Melhor modelo por F1-Score: {metrics_df.index[0]} ({metrics_df["F1-Score"].iloc[0]:.4f})')

metrics_df.to_csv('results/metricas_modelos.csv')
print('\nMetricas salvas em results/metricas_modelos.csv')

COMPARACAO DE METRICAS DOS MODELOS
                     Accuracy  Precision  Recall  F1-Score  AUC-ROC
Gradient Boosting      0.8908     0.5946  0.6875    0.6377   0.8614
XGBoost                0.8865     0.5789  0.6875    0.6286   0.8907
Random Forest          0.8777     0.5556  0.6250    0.5882   0.9067
Logistic Regression    0.7904     0.3621  0.6562    0.4667   0.8484

>>> Melhor modelo por F1-Score: Gradient Boosting (0.6377)

Metricas salvas em results/metricas_modelos.csv


In [ ]:
# 5.2 Visualizacao comparativa das metricas
fig, ax = plt.subplots(figsize=(14, 6))

x = np.arange(len(metrics_df.index))
width = 0.15
metrics_list = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC']
colors = ['#3498db', '#e67e22', '#27ae60', '#e74c3c', '#9b59b6']

for i, (metric, color) in enumerate(zip(metrics_list, colors)):
    bars = ax.bar(x + i * width, metrics_df[metric], width, label=metric, color=color, edgecolor='black', alpha=0.85)
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.005,
                f'{height:.3f}', ha='center', va='bottom', fontsize=8, rotation=45)

ax.set_xlabel('Modelo', fontsize=13)
ax.set_ylabel('Score', fontsize=13)
ax.set_title('Comparacao de Metricas entre Modelos', fontsize=14, fontweight='bold')
ax.set_xticks(x + width * 2)
ax.set_xticklabels(metrics_df.index, fontsize=11)
ax.legend(fontsize=10, loc='lower right')
ax.set_ylim(0, 1.15)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('results/08_comparacao_metricas.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafico salvo em results/08_comparacao_metricas.png')

Grafico salvo em results/08_comparacao_metricas.png


In [ ]:
# 5.3 Matrizes de Confusao
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

for i, (name, res) in enumerate(results.items()):
    ax = axes[i]
    cm = confusion_matrix(y_test, res['y_pred'])
    disp = ConfusionMatrixDisplay(cm, display_labels=['Baixa/Media', 'Alta'])
    disp.plot(ax=ax, cmap='Blues', values_format='d')
    ax.set_title(f'{name}', fontsize=13, fontweight='bold')

plt.suptitle('Matrizes de Confusao', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('results/09_matrizes_confusao.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafico salvo em results/09_matrizes_confusao.png')

Grafico salvo em results/09_matrizes_confusao.png


In [ ]:
# 5.4 Curvas ROC
from sklearn.metrics import roc_curve

fig, ax = plt.subplots(figsize=(10, 8))

colors_roc = ['#3498db', '#e74c3c', '#27ae60', '#9b59b6']
for (name, res), color in zip(results.items(), colors_roc):
    fpr, tpr, _ = roc_curve(y_test, res['y_proba'])
    auc = res['metrics']['AUC-ROC']
    ax.plot(fpr, tpr, color=color, linewidth=2.5, label=f'{name} (AUC = {auc:.4f})')

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5, label='Classificador Aleatorio')
ax.set_xlabel('Taxa de Falsos Positivos (FPR)', fontsize=13)
ax.set_ylabel('Taxa de Verdadeiros Positivos (TPR)', fontsize=13)
ax.set_title('Curvas ROC - Comparacao entre Modelos', fontsize=14, fontweight='bold')
ax.legend(fontsize=11, loc='lower right')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('results/10_curvas_roc.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafico salvo em results/10_curvas_roc.png')

Grafico salvo em results/10_curvas_roc.png


In [ ]:
# 5.5 Classification Report detalhado do melhor modelo
from sklearn.metrics import classification_report

best_model_name = metrics_df.index[0]
best_result = results[best_model_name]

print('=' * 70)
print(f'CLASSIFICATION REPORT - {best_model_name} (Melhor Modelo)')
print('=' * 70)
print(classification_report(
    y_test, best_result['y_pred'],
    target_names=['Baixa/Media Qualidade', 'Alta Qualidade']
))

CLASSIFICATION REPORT - Gradient Boosting (Melhor Modelo)
                       precision    recall  f1-score   support

Baixa/Media Qualidade       0.95      0.92      0.94       197
       Alta Qualidade       0.59      0.69      0.64        32

             accuracy                           0.89       229
            macro avg       0.77      0.81      0.79       229
         weighted avg       0.90      0.89      0.89       229



---

# 6. Interpretacao dos Resultados

Analise das variaveis mais influentes e implicacoes para a producao de vinhos.


In [ ]:
# 6.1 Feature Importance - Todos os modelos
fig, axes = plt.subplots(2, 2, figsize=(16, 14))
axes = axes.flatten()

for i, (name, model) in enumerate(trained_models.items()):
    ax = axes[i]
    feat_imp = get_feature_importance(model, X.columns.tolist())
    if feat_imp is not None:
        colors_fi = plt.cm.RdYlGn(np.linspace(0.2, 0.8, len(feat_imp)))
        ax.barh(feat_imp['Feature'], feat_imp['Importance'], color=colors_fi, edgecolor='black', alpha=0.85)
        ax.set_title(f'{name}', fontsize=13, fontweight='bold')
        ax.set_xlabel('Importancia', fontsize=11)
    else:
        ax.text(0.5, 0.5, 'N/A', ha='center', va='center', fontsize=16)
        ax.set_title(f'{name}', fontsize=13, fontweight='bold')

plt.suptitle('Importancia das Features por Modelo', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('results/11_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafico salvo em results/11_feature_importance.png')

Grafico salvo em results/11_feature_importance.png


In [ ]:
# 6.2 Top features do melhor modelo
best_model = trained_models[best_model_name]
feat_imp = get_feature_importance(best_model, X.columns.tolist())

print('=' * 70)
print(f'TOP FEATURES - {best_model_name}')
print('=' * 70)
if feat_imp is not None:
    for idx, row in feat_imp.iterrows():
        bar = '#' * int(row['Importance'] * 100)
        print(f'  {row["Feature"]:25s} | {row["Importance"]:.4f} | {bar}')

fig, ax = plt.subplots(figsize=(10, 7))
if feat_imp is not None:
    colors_top = plt.cm.RdYlGn(np.linspace(0.2, 0.8, len(feat_imp)))[::-1]
    bars = ax.barh(feat_imp['Feature'][::-1], feat_imp['Importance'][::-1],
                   color=colors_top, edgecolor='black', alpha=0.85)
    ax.set_xlabel('Importancia', fontsize=13)
    ax.set_title(f'Importancia das Features - {best_model_name}', fontsize=14, fontweight='bold')
    for bar, val in zip(bars, feat_imp['Importance'][::-1]):
        ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
                f'{val:.4f}', va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('results/12_top_features_melhor_modelo.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafico salvo em results/12_top_features_melhor_modelo.png')

TOP FEATURES - Gradient Boosting
  alcohol                   | 0.3816 | ######################################
  sulphates                 | 0.1693 | ################
  volatile acidity          | 0.1099 | ##########
  citric acid               | 0.0910 | #########
  total sulfur dioxide      | 0.0620 | ######
  fixed acidity             | 0.0601 | ######
  pH                        | 0.0390 | ###
  density                   | 0.0289 | ##
  free sulfur dioxide       | 0.0218 | ##
  residual sugar            | 0.0194 | #
  chlorides                 | 0.0171 | #


Grafico salvo em results/12_top_features_melhor_modelo.png


In [ ]:
# 6.3 Implicacoes para o processo de producao
print('=' * 70)
print('IMPLICACOES PARA O PROCESSO DE PRODUCAO')
print('=' * 70)
implicacoes = [
    '',
    'Com base na analise dos modelos, as seguintes variaveis demonstraram maior',
    'influencia na determinacao da qualidade do vinho:',
    '',
    '1. ALCOHOL (Teor Alcoolico)',
    '   - Variavel com maior correlacao positiva com a qualidade.',
    '   - Vinhos de alta qualidade tendem a ter maior teor alcoolico.',
    '   - Implicacao: Monitorar a fermentacao para atingir niveis ideais de alcool.',
    '',
    '2. VOLATILE ACIDITY (Acidez Volatil)',
    '   - Forte correlacao negativa com a qualidade.',
    '   - Altos niveis de acidez volatil indicam vinhos de menor qualidade.',
    '   - Implicacao: Controlar rigorosamente a acidez volatil durante a fermentacao',
    '     para evitar a formacao excessiva de acido acetico.',
    '',
    '3. SULPHATES (Sulfatos)',
    '   - Correlacao positiva com a qualidade.',
    '   - Sulfatos contribuem como antioxidante e antimicrobiano.',
    '   - Implicacao: Adicionar sulfatos em niveis adequados para preservacao.',
    '',
    '4. CITRIC ACID (Acido Citrico)',
    '   - Correlacao positiva - contribui para o frescor e sabor do vinho.',
    '   - Implicacao: Manter niveis balanceados para perfil sensorial agradavel.',
    '',
    '5. TOTAL SULFUR DIOXIDE (Dioxido de Enxofre Total)',
    '   - Correlacao negativa - excesso prejudica a qualidade.',
    '   - Implicacao: Dosar cuidadosamente o SO2 para equilibrar preservacao',
    '     e qualidade sensorial.',
    '',
    'CONCLUSAO GERAL:',
    'Os modelos de ML demonstraram capacidade de prever a qualidade dos vinhos',
    'com boa acuracia. O modelo pode ser utilizado como ferramenta complementar',
    'na linha de producao para triagem rapida de qualidade.',
]
for line in implicacoes:
    print(line)

IMPLICACOES PARA O PROCESSO DE PRODUCAO

Com base na analise dos modelos, as seguintes variaveis demonstraram maior
influencia na determinacao da qualidade do vinho:

1. ALCOHOL (Teor Alcoolico)
   - Variavel com maior correlacao positiva com a qualidade.
   - Vinhos de alta qualidade tendem a ter maior teor alcoolico.
   - Implicacao: Monitorar a fermentacao para atingir niveis ideais de alcool.

2. VOLATILE ACIDITY (Acidez Volatil)
   - Forte correlacao negativa com a qualidade.
   - Altos niveis de acidez volatil indicam vinhos de menor qualidade.
   - Implicacao: Controlar rigorosamente a acidez volatil durante a fermentacao
     para evitar a formacao excessiva de acido acetico.

3. SULPHATES (Sulfatos)
   - Correlacao positiva com a qualidade.
   - Sulfatos contribuem como antioxidante e antimicrobiano.
   - Implicacao: Adicionar sulfatos em niveis adequados para preservacao.

4. CITRIC ACID (Acido Citrico)
   - Correlacao positiva - contribui para o frescor e sabor do vinho.
   

---

# Conclusao

Este projeto demonstrou a viabilidade de utilizar modelos de Machine Learning para classificar a qualidade de vinhos tintos a partir de variaveis fisico-quimicas.

**Principais achados:**
- O **desbalanceamento de classes** (86% vs 14%) foi tratado com SMOTE, melhorando significativamente o recall da classe minoritaria
- As variaveis **alcohol**, **volatile acidity** e **sulphates** foram consistentemente identificadas como as mais importantes para a classificacao
- Os modelos baseados em **ensemble** (Random Forest, XGBoost, Gradient Boosting) superaram o modelo linear (Logistic Regression) na maioria das metricas
- A **validacao cruzada** confirmou a robustez dos resultados, com baixa variancia entre folds

**Limitacoes:**
- Dataset relativamente pequeno (1.143 amostras)
- Somente vinhos tintos - resultados podem nao generalizar para brancos
- Classificacao binaria simplifica as nuances da qualidade

**Proximos passos sugeridos:**
- Testar com threshold diferente (ex: >= 6 como alta qualidade)
- Aplicar feature engineering mais avancada (interacoes entre variaveis)
- Experimentar Deep Learning com mais dados
- Incluir variaveis sensoriais quando disponiveis
